In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
import warnings
warnings.filterwarnings('ignore')

### After initial EDA, we check for significant features for modelling

It can be done using different methods 
1. Correlation Analysis and Mutual Information
2. Statistical Tests:Annova, Chi Square Test
3. Feature Selection Techniques:
    - Forward Selection: Starts with no variables in the model, adding them one by one, testing at each step if the added variable significantly improves the model fit.
    - Backward Elimination: Starts with all potential variables, removing them one by one, keeping those that contribute significantly to the model.
    - Recursive Feature Elimination (RFE): An iterative process that starts with all variables and eliminates the least important variable at each step.
4. Feature Importance based on ML Algorithm (Random Forest, XGBoost) 

So lets start with visually checking the variables

In [ ]:
plt.figure(figsize=(12, 6))
for cols in train.columns[6:]:
    plt.scatter(train[cols],train['Target'], label='Series 1')
    plt.xlabel(cols)
    plt.ylabel('Target')
    plt.title(f'Scatter Plot of {cols} vs Target')
    plt.show()

In [ ]:
from sklearn.feature_selection import mutual_info_regression

def correl_table(df,features, target):
    corr_pearson = df[features].corrwith(df[target], method='pearson')
    corr_spearman = df[features].corrwith(df[target], method='spearman')

    df_no_nan = df.dropna()
    mi_scores = {}
    for col in features:
        mi_scores[col] = mutual_info_regression(df_no_nan[[col]], df_no_nan[target])[0]

    correlation_target = pd.DataFrame({'Pearson_correlation': corr_pearson, 
                                      'Spearman_correlation': corr_spearman,
                                      'Mutual_Information': mi_scores})
    return correlation_target.reset_index()

In [ ]:
# List of numeric columns
numeric_columns = train.select_dtypes(include=['number']).columns.tolist()

# List of feature columns
features = numeric_columns

correlation_Target = correl_table(train,features, 'Target')

### Statistical Models

1. Linear Regression
2. Ridge and Lasso Regression
3. GLMs (Gamma, Poisson, NB)

If we are using Statistical models, then we must check for multicolinearity as well.
There are other code block with are required in Model building like Train Test Split, Missing value Treatment, Outlier Treatment

For all these codes, refer Classification Modelling Jupyter notebook

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import glm

In [ ]:
# Fit Linear Regression model
X = sm.add_constant(train[features])
y = train['Target']
lm_model = sm.OLS(y, X).fit()
print(lm_model.summary())

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
y_pred_lm = lm_model.fittedvalues
y_test = train['Target']

rmse = np.sqrt(mean_squared_error(y_test, y_pred_lm))
print(f"Train Root Mean Squared Error: {rmse}")

In [ ]:
y_pred_lm = lm_model.predict(validation[stats_features_lm])
y_test = validation['Target']

rmse = np.sqrt(mean_squared_error(y_test, y_pred_lm))
print(f"Test Root Mean Squared Error: {rmse}")

In [ ]:
# For recusrsive feature elimination
p_value = lm_model.pvalues.iloc[1:].max()
while p_value > 0.05:
    var = list(pd.DataFrame(lm_model.pvalues.iloc[1:]).sort_values(by = 0, ascending= False).reset_index().drop(0)['index'])
    # Fit Linear Regression model
    lm_model = sm.OLS(train['Target'], sm.add_constant(train[var])).fit()
    print(lm_model.summary())
    p_value = lm_model.pvalues.iloc[1:].max()

In [ ]:
# Ridge
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=0.1)  # alpha is the regularization strength
ridge.fit(train[features_1], train['Target'])

In [ ]:
coefficients = ridge.coef_

coefficients_df = pd.DataFrame({
    'Feature': features_1,
    'Coefficient': coefficients
})

# Display the DataFrame
coefficients_df

In [ ]:
#Lasso
from sklearn.linear_model import Lasso
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(train[features], train['Target'])

In [ ]:
coefficients = lasso.coef_

coefficients_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': coefficients
})

# Display the DataFrame
coefficients_df

In [ ]:
# Fit GLM - Gamma model
X = sm.add_constant(train[features])
y = train['Target'] 

glm_model = sm.GLM(y, X, family=sm.families.Gamma()).fit()
print(glm_model.summary())

In [ ]:
# Recursive Feature elimination
p_value = glm_model.pvalues.iloc[1:].max()
while p_value > 0.1:
    var = list(pd.DataFrame(glm_model.pvalues.iloc[1:]).sort_values(by = 0, ascending= False).reset_index().drop(0)['index'])
    # Fit GLM model
    glm_model = sm.GLM(train['Target']+10, sm.add_constant(train[var]), family=sm.families.Gamma()).fit()
    print(glm_model.summary())
    p_value = glm_model.pvalues.iloc[1:].max()

For GLMs, we can use different family distribution based on the Target Variable. Also for evaluating the performance of the model. We can use different metrics like
    
    1. RMSE
    
    2. MAPE
    
    3. MAE

Depending on the application, I personally prefer RMSE or MAPE 

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Random Forest Regression
forest_model = RandomForestRegressor(n_estimators=100, random_state=42)
forest_model.fit(train[features_tree], train['Target'])

y_test = train['Target']
y_pred = forest_model.predict(train[features_tree])
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f" Root Mean Squared Error: {rmse}")
print(rmse/np.std(y_test))



y_test = validation['Target']
y_pred = forest_model.predict(validation[features_tree])
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f" Root Mean Squared Error: {rmse}")
print(rmse/np.std(y_test))



importances = forest_model.feature_importances_
feature_names = features_tree

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sort the DataFrame by importance
importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df

In [ ]:
### Manual HyperParameter Tuning

y_test = validation['Target']
# Random Forest Regression

for j in [True , False]:
    for k in [None,3,5,6,7,8,9 ,10]:
        for m in [2,3,4,5,10]: 
            for n in [2,3,4, 5, 10]: 

                forest_model = RandomForestRegressor(
                    n_estimators=100,
                    bootstrap=j,
                    max_depth=k,
                    min_samples_leaf=m,
                    min_samples_split=n,
                    random_state=42
                )

                print(forest_model)
                print('bootstrap:',j)
                print('max_depth:',k)
                forest_model.fit(train[features_tree], train['Target'])
                
                
                y_test = train['Target']
                y_pred = forest_model.predict(train[features_tree])
                rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                r2 = r2_score(y_test, y_pred)
                print(f" Root Mean Squared Error: {rmse}")
                print(rmse/np.std(y_test))



                y_test = validation['Target']
                y_pred = forest_model.predict(validation[features_tree])
                rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                r2 = r2_score(y_test, y_pred)
                print(f" Root Mean Squared Error: {rmse}")
                print(rmse/np.std(y_test))

In [ ]:
### Using GRID Search
from sklearn.model_selection import GridSearchCV
# Define the model
rf = RandomForestRegressor(random_state=42)

# Set up the grid of parameters
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit GridSearchCV
grid_search.fit(train[features_tree], train['Target'])

# Best parameters and best score
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

In [ ]:
### Using Random Search
from sklearn.model_selection import RandomizedSearchCV

# Define the model
rf = RandomForestRegressor(random_state=42)

# Define the parameter grid
param_distributions = {
    'n_estimators': np.arange(100, 501, 50),
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': np.arange(2, 21, 2),
    'min_samples_leaf': np.arange(1, 11, 1)
}

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_distributions, n_iter=100, cv=5, n_jobs=-1, verbose=2, random_state=42)

# Fit RandomizedSearchCV
random_search.fit(train[features_tree], train['Target'])

# Best parameters and best score
print("Best parameters:", random_search.best_params_)
print("Best score:", random_search.best_score_)

### XGBoost

In [ ]:
# XGB Regression
from xgboost import XGBRegressor

model_xgb = XGBRegressor()

model_xgb.fit(train[features_xgb], train['Target'])

y_test = train['Target']
y_pred = model_xgb.predict(train[features_xgb])
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f" Root Mean Squared Error: {rmse}")
print(rmse/np.std(y_test))

y_test = validation['Target']
y_pred = model_xgb.predict(validation[features_xgb])
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f" Root Mean Squared Error: {rmse}")
print(rmse/np.std(y_test))



importances = model_xgb.fea
ture_importances_
feature_names = features_xgb

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sort the DataFrame by importance
importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df

In [ ]:
# Manual HyperParameter Tuning

for i in [1, 5, 10]:
    for j in [None,3,4,5]:
        for k in [0.6, 0.8, 1.0]: 
            for l in [0.6, 0.8, 1.0]: 
                for m in [0.5, 1, 1.5, 2, 5]:

                    model_xgb = XGBRegressor(
                        n_estimators=100,
                        max_depth=j,
                        min_child_weight=i,
                        subsample=k,
                        colsample_bytree = l,
                        gamma = m,
                        random_state=42
                    )

                    print(model_xgb)
                    print('min_child_weight:',i)
                    print('max_depth:',j)
                    print('subsample:',k)
                    print('colsample_bytree:',l)
                    print('gamma:',m)
                    model_xgb.fit(train[features_xgb], train['Target'])
                    
                    y_test = train['Target']
                    y_pred = model_xgb.predict(train[features_xgb])
                    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                    r2 = r2_score(y_test, y_pred)
                    print(f"Train Root Mean Squared Error: {rmse}")
                    print(rmse/np.std(y_test))

                    y_test = validation['Target']
                    y_pred = model_xgb.predict(validation[features_xgb])
                    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                    r2 = r2_score(y_test, y_pred)
                    print(f"Test Root Mean Squared Error: {rmse}")
                    print(rmse/np.std(y_test))

In [ ]:
### Using GRID Search
# Define the model
xg_reg = xgb.XGBRegressor(objective ='reg:squarederror', seed=42)

# Create the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'colsample_bytree': [0.3, 0.7]
}

# Setup GridSearchCV
grid_search = GridSearchCV(estimator=xg_reg, param_grid=param_grid, scoring='neg_mean_squared_error', cv=3, verbose=1)

# Fit GridSearchCV
grid_search.fit(train[features_xgb], train['Target'])

# Print best parameters and results
print("Best parameters found: ", grid_search.best_params_)
print("Best score found: ", grid_search.best_score_)

In [ ]:
### Using Random Search
# Define the model
xg_reg = xgb.XGBRegressor(objective ='reg:squarederror', seed=42)

# Define the parameter distributions
param_distributions = {
    'n_estimators': np.arange(100, 1001, 100),
    'learning_rate': np.linspace(0.01, 0.2, 20),
    'max_depth': np.arange(3, 10),
    'colsample_bytree': np.linspace(0.3, 1.0, 8)
}

# Setup RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=xg_reg, param_distributions=param_distributions, n_iter=100, scoring='neg_mean_squared_error', cv=3, verbose=1, random_state=42)

# Fit RandomizedSearchCV
random_search.fit(train[features_xgb], train['Target'])

# Print best parameters and results
print("Best parameters found: ", random_search.best_params_)
print("Best score found: ", random_search.best_score_)

In [ ]:
### Neural Networks - Simple Neural Using PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.optim as optim

features_nn = []

In [ ]:
# Convert data to PyTorch tensors
X_tensor = torch.tensor(train[features_nn].values, dtype=torch.float32)
y_tensor = torch.tensor(train['Target'].values, dtype=torch.float32).view(-1, 1)  # Reshape to column vector

# Create dataset and dataloaders
dataset = TensorDataset(X_tensor, y_tensor)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Define the Neural Network Model
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super(SimpleNN, self).__init__()
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.layer3 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = torch.relu(self.layer3(x))
        x = self.output(x)
        return x

In [ ]:
# Initialize the model, loss function, and optimizer
model = SimpleNN(input_dim=X_tensor.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the Model
n_epochs = 100

for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        # Forward pass
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}')

In [ ]:
# Evaluate the Model
model.eval()
with torch.no_grad():
    train_predictions = model(X_tensor[train_dataset.indices])
    test_predictions = model(X_tensor[test_dataset.indices])
    train_mse = criterion(train_predictions, y_tensor[train_dataset.indices]).item()
    test_mse = criterion(test_predictions, y_tensor[test_dataset.indices]).item()

print(f'Train MSE: {train_mse:.4f}')
print(f'Test MSE: {test_mse:.4f}')

In [ ]:
# Predicting any outputs
with torch.no_grad():
    sample_data = torch.tensor([X_test[0]], dtype=torch.float32)  # Use the first test sample
    predicted = model(sample_data)
    print(f'Predicted value: {predicted.item()} Actual value: {y_test[0].item()}')

In [ ]:
#If you want to try manual predictions


### Change the dictionary based on the network architecture
weights = {
    'layer1_weights': model.layer1.weight.data.numpy(),
    'layer1_bias': model.layer1.bias.data.numpy(),
    'layer2_weights': model.layer2.weight.data.numpy(),
    'layer2_bias': model.layer2.bias.data.numpy(),
    'layer3_weights': model.layer3.weight.data.numpy(),
    'layer3_bias': model.layer3.bias.data.numpy(),
    'output_weights': model.output.weight.data.numpy(),
    'output_bias': model.output.bias.data.numpy()
}

def manual_predict(X, weights):
    # Forward pass through the network manually
    layer1_output = np.maximum(0, np.dot(X, weights['layer1_weights'].T) + weights['layer1_bias'])  # ReLU activation
    layer2_output = np.maximum(0, np.dot(layer1_output, weights['layer2_weights'].T) + weights['layer2_bias'])  # ReLU activation
    layer3_output = np.maximum(0, np.dot(layer2_output, weights['layer3_weights'].T) + weights['layer3_bias'])  # ReLU activation
    output = np.dot(layer3_output, weights['output_weights'].T) + weights['output_bias']  # Linear activation for regression
    return output

manual_predict(validation[features_nn].values, weights)